notebook para testar a geracao de um df a apartir de uma
matriz (imagem) e executar o pca

# Imports

In [1]:
import os
import gc # to free memory
import numpy as np
from math import ceil
import pandas as pd
import psutil
import time
import sys
import glob

import pickle
import random
from tqdm import tqdm

import multiprocessing
nproc = multiprocessing.cpu_count()-2

In [3]:
# import functions
#from functions.functions_pca import list_files_to_read, get_bandsDates, gen_dfToPCA_filter
%run ../code/functions/functions_pca

In [2]:
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pyspark.pandas as ps

In [4]:
#imports for spark
from pyspark.sql import SparkSession

In [5]:
#imports for spark, cont.
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, array, udf
from pyspark.ml.feature import PCA
from pyspark.ml.feature import PCAModel

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors

In [6]:
# Step 1: Start a Spark ssession
spark = SparkSession.builder \
    .appName("PySpark to PCA") \
    .config("spark.driver.memory", "9g") \
    .getOrCreate()

24/08/13 18:32:33 WARN Utils: Your hostname, MacBook-Air-M2.local resolves to a loopback address: 127.0.0.1; using 139.82.120.42 instead (on interface en0)
24/08/13 18:32:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/13 18:32:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Code

## Gen df from matrix

In [8]:

#read band image files
all_bands = ['B04', 'B03', 'B02', 'B08', 'EVI', 'NDVI']
read_dir = '/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/data/Cassio/S2-16D_V2_012014_20220728_/'
name_img = 'S2-16D_V2_012014'
band_tile_img_files = list_files_to_read(read_dir, name_img)
bands, dates= get_bandsDates(band_tile_img_files, tile=1)
cur_dir = os.getcwd()

4.100799560546875e-05


In [9]:
bands, dates

(['B08', 'B04', 'B02', 'EVI', 'B03', 'NDVI'], ['20220728', '20220829'])

In [10]:
#read image files
#teste para gerar spark df da matrix
t='20220728'
band_img_files=band_tile_img_files
band_img_file_to_load=[x for x in band_img_files if t in x]
image_band_dic = {}
image_band_dic = load_image_files3(band_img_file_to_load, pos=-1)
bands = image_band_dic.keys()
print ( bands)#, band_img_file_to_load)

dict_keys(['B08', 'B04', 'B02', 'EVI', 'B03', 'NDVI'])


In [17]:
img_bands = np.dstack([image_band_dic[x] for x in list(bands)[:2]])
cols_name = [x+'_'+t for x in list(bands)[:2]]
cols_name

['B08_20220728', 'B04_20220728']

In [18]:
#testar a funcao gen_sdf_from_img_band aqui
sh_print=1
num_rows, num_cols, cols_df = img_bands.shape
print (f'matrix rows={num_rows}, cols={num_cols}, bands={cols_df}') if sh_print else None


matrix rows=10560, cols=10560, bands=2


In [19]:
# Flatten the matrix values and reshape into a 2D array
values = img_bands.reshape(-1, cols_df)

In [20]:
values.shape

(111513600, 2)

In [21]:
# Create a spark DataFrame from the index positions and values
t1=time.time()
sdft = ps.DataFrame(values, columns=cols_name)
t2=time.time()
t2-t1

1679.0067551136017

In [22]:
sdft.shape, (t2-t1)/60

24/08/13 19:11:30 WARN TaskSetManager: Stage 0 contains a task of very large size (174474 KiB). The maximum recommended task size is 1000 KiB.


((111513600, 2), 27.98344591856003)

In [17]:
sdft.head()

24/08/13 17:42:03 WARN TaskSetManager: Stage 3 contains a task of very large size (174474 KiB). The maximum recommended task size is 1000 KiB.
24/08/13 17:42:07 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 3 (TID 9): Attempting to kill Python Worker


,B08_20220728,B04_20220728
0,3296,274
1,3404,260
2,3466,253
3,3237,253
4,3040,250


In [23]:
# Create a list of index positions [i, j]
pos_0=np.repeat(np.arange(img_bands.shape[0]), img_bands.shape[1])
pos_1=np.tile(np.arange(img_bands.shape[1]), img_bands.shape[0])
#pyspark pandas dataframe doesn't support ndarray
pos_0 = pos_0.tolist()
pos_1 = pos_1.tolist()
type(pos_0)

list

In [27]:
#from math import ceil
n = ceil(sys.getsizeof(pos_0) / 102400)
n, len(pos_0)

(8713, 111513600)

In [20]:
sdft.insert(0,'coords_0', pos_0)
sdft.insert(1,'coords_1', pos_1)

24/08/13 17:42:46 WARN TaskSetManager: Stage 4 contains a task of very large size (174474 KiB). The maximum recommended task size is 1000 KiB.
24/08/13 17:50:56 WARN TaskSetManager: Stage 7 contains a task of very large size (174474 KiB). The maximum recommended task size is 1000 KiB.
24/08/13 17:51:36 WARN TaskSetManager: Stage 8 contains a task of very large size (174474 KiB). The maximum recommended task size is 1000 KiB.
24/08/13 17:52:14 WARN TaskSetManager: Stage 9 contains a task of very large size (133731 KiB). The maximum recommended task size is 1000 KiB.
24/08/13 17:52:51 WARN AttachDistributedSequenceExec: clean up cached RDD(23) in AttachDistributedSequenceExec(146)
24/08/13 17:53:08 WARN TaskMemoryManager: Failed to allocate a page (67108864 bytes), try again.
24/08/13 17:53:09 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
24/08/13 17:53:10 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
24/08/13 17:53:10 WAR

In [28]:
sdft['coords_0'] = pos_0

ConnectionRefusedError: [Errno 61] Connection refused

In [21]:
sdft.head()

24/08/13 18:09:56 WARN TaskSetManager: Stage 17 contains a task of very large size (174474 KiB). The maximum recommended task size is 1000 KiB.
24/08/13 18:10:04 ERROR Executor: Exception in task 6.0 in stage 17.0 (TID 68)7]
java.lang.OutOfMemoryError: Java heap space
24/08/13 18:10:04 ERROR Executor: Exception in task 4.0 in stage 17.0 (TID 66)
java.lang.OutOfMemoryError: Java heap space
24/08/13 18:10:04 ERROR Executor: Exception in task 3.0 in stage 17.0 (TID 65)
java.lang.OutOfMemoryError: Java heap space
24/08/13 18:10:04 ERROR Executor: Exception in task 2.0 in stage 17.0 (TID 64)
java.lang.OutOfMemoryError: Java heap space
24/08/13 18:10:04 ERROR Executor: Exception in task 5.0 in stage 17.0 (TID 67)
java.lang.OutOfMemoryError: Java heap space
24/08/13 18:10:04 ERROR Executor: Exception in task 1.0 in stage 17.0 (TID 63)
java.lang.OutOfMemoryError: Java heap space
24/08/13 18:10:04 ERROR Executor: Exception in task 0.0 in stage 17.0 (TID 62)
java.lang.OutOfMemoryError: Java heap

ConnectionRefusedError: [Errno 61] Connection refused

ConnectionRefusedError: [Errno 61] Connection refused

## Gen spark df to PCA

In [22]:
# Convert to PySpark DataFrame
spark_df = sdftM.to_spark()

/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/.venv_data_geobia/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [23]:
spark_df.show()

24/08/13 16:22:48 WARN AttachDistributedSequenceExec: clean up cached RDD(64) in AttachDistributedSequenceExec(828)


+--------+--------+-------+--------+----+-------+
|coords_0|coords_1|     b1|      b2|  b3|     b4|
+--------+--------+-------+--------+----+-------+
|       0|       0|-9999.0|     2.0| 3.0|    4.0|
|       0|       1|    5.0|     6.0|-7.0|    8.0|
|       0|       2|    9.0|    10.0|11.0|-9999.0|
|       1|       0|   13.0|    14.0|15.0|   16.0|
|       1|       1|   17.0|    18.0|19.0|   20.0|
|       1|       2|   21.0|    22.0|23.0|   24.0|
|       2|       0|   25.0|    26.0|27.0|   28.0|
|       2|       1|   29.0|    30.0|31.0|   32.0|
|       2|       2|   33.0|-32768.0|35.0|   36.0|
+--------+--------+-------+--------+----+-------+



In [26]:
spark_df_coords = spark_df
#%% #replace nans
t1=time.time()
# Replace -9999 with NaN in all columns
spark_df = spark_df.select([when(col(c) == -9999, F.lit(None)).otherwise(col(c)).alias(c) for c in spark_df.columns])
spark_df = spark_df.select([when(col(c) == -32768, F.lit(None)).otherwise(col(c)).alias(c) for c in spark_df.columns])
t2=time.time()
t2-t1

0.303272008895874

In [27]:
# #drop the nans
spark_df = spark_df.na.drop()
spark_df.show()

24/08/13 16:33:09 WARN AttachDistributedSequenceExec: clean up cached RDD(84) in AttachDistributedSequenceExec(1248)


+--------+--------+----+----+----+----+
|coords_0|coords_1|  b1|  b2|  b3|  b4|
+--------+--------+----+----+----+----+
|       0|       1| 5.0| 6.0|-7.0| 8.0|
|       1|       0|13.0|14.0|15.0|16.0|
|       1|       1|17.0|18.0|19.0|20.0|
|       1|       2|21.0|22.0|23.0|24.0|
|       2|       0|25.0|26.0|27.0|28.0|
|       2|       1|29.0|30.0|31.0|32.0|
+--------+--------+----+----+----+----+



In [28]:
# assembler nao suporta nan's
assembler = VectorAssembler(inputCols=bands_name, outputCol="features")
df_with_features=assembler.transform(spark_df)
print (df_with_features.show())
df_with_features.dtypes

24/08/13 16:36:20 WARN AttachDistributedSequenceExec: clean up cached RDD(104) in AttachDistributedSequenceExec(1645)


+--------+--------+----+----+----+----+--------------------+
|coords_0|coords_1|  b1|  b2|  b3|  b4|            features|
+--------+--------+----+----+----+----+--------------------+
|       0|       1| 5.0| 6.0|-7.0| 8.0|  [5.0,6.0,-7.0,8.0]|
|       1|       0|13.0|14.0|15.0|16.0|[13.0,14.0,15.0,1...|
|       1|       1|17.0|18.0|19.0|20.0|[17.0,18.0,19.0,2...|
|       1|       2|21.0|22.0|23.0|24.0|[21.0,22.0,23.0,2...|
|       2|       0|25.0|26.0|27.0|28.0|[25.0,26.0,27.0,2...|
|       2|       1|29.0|30.0|31.0|32.0|[29.0,30.0,31.0,3...|
+--------+--------+----+----+----+----+--------------------+

None


[('coords_0', 'bigint'),
 ('coords_1', 'bigint'),
 ('b1', 'double'),
 ('b2', 'double'),
 ('b3', 'double'),
 ('b4', 'double'),
 ('features', 'vector')]

## Run PCA

In [29]:
# Apply PCA
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(df_with_features)
df_with_pca = pca_model.transform(df_with_features)
df_with_pca.show()

24/08/13 16:36:39 WARN AttachDistributedSequenceExec: clean up cached RDD(124) in AttachDistributedSequenceExec(2071)
24/08/13 16:36:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/08/13 16:36:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
24/08/13 16:36:43 WARN AttachDistributedSequenceExec: clean up cached RDD(151) in AttachDistributedSequenceExec(2472)


+--------+--------+----+----+----+----+--------------------+--------------------+
|coords_0|coords_1|  b1|  b2|  b3|  b4|            features|        pca_features|
+--------+--------+----+----+----+----+--------------------+--------------------+
|       0|       1| 5.0| 6.0|-7.0| 8.0|  [5.0,6.0,-7.0,8.0]|[-3.4876489974979...|
|       1|       0|13.0|14.0|15.0|16.0|[13.0,14.0,15.0,1...|[-28.494790964050...|
|       1|       1|17.0|18.0|19.0|20.0|[17.0,18.0,19.0,2...|[-36.322551623143...|
|       1|       2|21.0|22.0|23.0|24.0|[21.0,22.0,23.0,2...|[-44.150312282235...|
|       2|       0|25.0|26.0|27.0|28.0|[25.0,26.0,27.0,2...|[-51.978072941328...|
|       2|       1|29.0|30.0|31.0|32.0|[29.0,30.0,31.0,3...|[-59.805833600420...|
+--------+--------+----+----+----+----+--------------------+--------------------+



In [30]:
# Show the PCA results
df_with_pca.select("pca_features").show(truncate=False)

24/08/13 16:36:51 WARN AttachDistributedSequenceExec: clean up cached RDD(171) in AttachDistributedSequenceExec(2902)


+-----------------------------------------+
|pca_features                             |
+-----------------------------------------+
|[-3.4876489974979212,-12.536731543890735]|
|[-28.494790964050924,-5.420352502224754] |
|[-36.322551623143305,-7.0714621484825795]|
|[-44.150312282235674,-8.722571794740407] |
|[-51.97807294132806,-10.373681440998233] |
|[-59.80583360042043,-12.024791087256059] |
+-----------------------------------------+



In [31]:
pca_model.explainedVariance

DenseVector([0.9806, 0.0194])

In [33]:
pca_model.transform(df_with_features).collect()[1].pca_features

24/08/13 16:38:54 WARN AttachDistributedSequenceExec: clean up cached RDD(211) in AttachDistributedSequenceExec(3792)


DenseVector([-28.4948, -5.4204])

In [30]:
#close sparky session
#spark.stop()